# 모르면 모른다고 답하는 챗봇 — 기준 점수

**「코랩으로 만드는 업무 챗봇」 17장 예제** — 지식베이스에 **검색 → 기준 점수 → 답 → 화면**을 차례로 붙여, 근거가 있을 때만 답하는 챗봇을 완성합니다.

- 걸리는 시간: 약 40분
- 키: 필요 없습니다(전부 키 없이 끝까지 돕니다).

### 시작하는 방법
1. 「Drive로 복사」를 누른 뒤 위에서부터 한 칸씩 실행합니다(칸 왼쪽 ▶ 또는 Shift+Enter).
2. 「준비」 아래의 칸들은 처음 한 번만 실행합니다. 설치는 1~2분 걸립니다.
3. 예제마다 **확인 포인트**와 **바꿔 보기**가 있습니다. 확인 포인트의 값이 화면과 같은지 보고, 바꿔 보기는 직접 고쳐 다시 실행합니다.

### 이 노트북의 환경
이 노트북은 Airflow 없이 돕니다. 검색과 화면에만 집중합니다. 코랩 런타임이 끊기면 처음부터 다시 실행하면 됩니다.

### 예제 구성
| 번호 | 예제 |
|---|---|
| 1 | 검색 결과를 눈으로 보기 |
| 2 | 기준 점수 정하기 — 문서 안 질문과 밖 질문 |
| 3 | 답하는 함수 — 기준 미달이면 AI 를 부르지 않는다 |
| 4 | 일부러 기준을 없애기 — 엉뚱한 근거가 붙는다 |
| 5 | 화면 붙이기 — 대화형 챗봇 |

## 준비

화면은 이 책의 다른 예제처럼 Gradio 로 만듭니다. 답의 흐름은 **질문 → 검색 → 기준 확인 → 답**입니다. 임베딩은 코랩 안에서 도는 다국어 모델이라 키가 필요 없습니다.

In [ ]:
# 준비 1 — 설치 (1분쯤, 처음 한 번만)
import importlib.util, os, subprocess, sys, time

os.environ["TZ"] = "Asia/Seoul"
time.tzset()
HOME = "/content"
os.makedirs(f"{HOME}/dags", exist_ok=True)

NEED = {"fastembed": "fastembed", "qdrant_client": "qdrant-client", "gradio": "gradio", "openai": "openai", "pandas": "pandas"}
for mod, pkg in NEED.items():
    if importlib.util.find_spec(mod) is None:
        print("설치:", pkg)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False, capture_output=True)
importlib.invalidate_caches()
missing = [m for m in NEED if importlib.util.find_spec(m) is None]
print("아직 없음 → " + ", ".join(missing) + " · 이 칸을 한 번 더 실행해 주세요." if missing else "설치 확인 완료")

준비 2 — 지식베이스 도구입니다. 앞 노트북(kb_01)에서 만든 `kb.py` 와 같은 파일입니다.

In [ ]:
%%writefile /content/dags/kb.py
# kb.py — 지식베이스 도구
import sys
import uuid
from pathlib import Path

LAKE = Path("/content/lake/raw-docs")       # Lake — 원본 문서를 그대로 두는 곳
QDRANT = "/content/qdrant"                  # VectorDB — Qdrant(파일로 저장)
COLLECTION = "lecture_kb"
MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # 384차원 · 키 없이 코랩 안에서 계산
DIM = 384
THRESHOLD = 0.40                            # 이 점수보다 낮으면 「찾지 못했습니다」 — kb_02 에서 재서 정한 값

_model = None


def embed(texts):
    global _model
    if _model is None:
        from fastembed import TextEmbedding
        _model = TextEmbedding(MODEL, cache_dir="/content/fastembed")
    return [v.tolist() for v in _model.embed(list(texts))]


def chunk(text, size=200, overlap=40):
    """문단을 이어 붙여 size 글자쯤 되면 한 조각 — 조각 끝 overlap 글자는 다음 조각 앞에 다시 넣습니다."""
    chunks, buf = [], ""
    for para in [p.strip() for p in text.split("\n\n") if p.strip()]:
        if buf and len(buf) + len(para) > size:
            chunks.append(buf)
            buf = buf[-overlap:]
        buf = (buf + " " + para).strip()
    if buf:
        chunks.append(buf)
    return chunks


def client():
    from qdrant_client import QdrantClient, models
    c = QdrantClient(path=QDRANT)
    if not c.collection_exists(COLLECTION):
        c.create_collection(COLLECTION, vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE))
    return c


def _of(source):
    from qdrant_client import models
    return models.Filter(must=[models.FieldCondition(key="source", match=models.MatchValue(value=source))])


def remove_doc(c, source):
    from qdrant_client import models
    c.delete(COLLECTION, points_selector=models.FilterSelector(filter=_of(source)))


def index_doc(c, source, text, extra=None):
    """문서 하나를 넣습니다 — 먼저 그 문서의 조각을 지우고, 「문서 이름#순번」으로 만든 id 로 다시 넣습니다(멱등)."""
    from qdrant_client import models
    remove_doc(c, source)
    chunks = chunk(text)
    points = [models.PointStruct(id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"{source}#{i}")), vector=v,
                                 payload={"source": source, "no": i, "text": t, **(extra or {})})
              for i, (t, v) in enumerate(zip(chunks, embed(chunks)))]
    c.upsert(COLLECTION, points=points)
    return len(points)


def stat(c):
    """문서별 조각 수"""
    out, offset = {}, None
    while True:
        points, offset = c.scroll(COLLECTION, limit=256, offset=offset, with_payload=True)
        for p in points:
            out[p.payload["source"]] = out.get(p.payload["source"], 0) + 1
        if offset is None:
            return dict(sorted(out.items()))


def search(question, k=3):
    c = client()
    try:
        hits = c.query_points(COLLECTION, query=embed([question])[0], limit=k, with_payload=True).points
        return [(round(h.score, 3), h.payload["source"], h.payload["text"]) for h in hits]
    finally:
        c.close()


def answer(question, threshold=THRESHOLD):
    """가장 가까운 조각의 점수가 기준보다 낮으면 답하지 않습니다."""
    hits = search(question)
    if not hits or hits[0][0] < threshold:
        top = hits[0][0] if hits else 0
        return f"찾지 못했습니다 (가장 가까운 조각의 점수 {top} < 기준 {threshold})"
    score, source, text = hits[0]
    return f"[근거 {source} · 점수 {score}]\n{text}"


if __name__ == "__main__":                  # 터미널에서: python kb.py stat | search "질문" | ask "질문" | reset
    cmd = sys.argv[1] if len(sys.argv) > 1 else "stat"
    if cmd == "reset":
        import shutil
        shutil.rmtree(QDRANT, ignore_errors=True)
        print("지식베이스를 비웠습니다")
    elif cmd == "stat":
        c = client()
        s = stat(c)
        c.close()
        print(f"조각 {sum(s.values())}개 · 문서 {len(s)}개")
        for name, n in s.items():
            print(f"  {name:20s} {n}조각")
    elif cmd == "search":
        for score, source, text in search(sys.argv[2]):
            print(f"  {score:<6} {source:18s} {text[:46]}…")
    elif cmd == "ask":
        print(answer(sys.argv[2], float(sys.argv[3]) if len(sys.argv) > 3 else THRESHOLD))

In [ ]:
# 준비 3 — 지식베이스 만들기 (kb_01 의 파이프라인이 만든 것과 같은 결과물)
import shutil, sys
sys.path.insert(0, "/content/dags")
import kb

DOCS = {
    "airflow_ops.txt": """Airflow 운영 안내

Airflow 는 여러 단계로 된 작업을 정해진 순서와 시각에 실행하는 오케스트레이션 도구입니다. 작업의 순서도는 DAG 라고 부르고 파이썬 파일로 적습니다.

스케줄러가 멈추면 웹 화면은 열리지만 실행이 queued 상태에 머뭅니다. 이때는 docker compose ps 로 스케줄러 컨테이너의 상태를 보고 docker compose logs airflow-scheduler 로 원인을 확인합니다.

태스크가 실패하면 retries 에 적은 횟수만큼 다시 시도합니다. 그래도 실패하면 뒤 태스크는 upstream_failed 가 되어 시작하지 않습니다. 원인을 고친 뒤 실패한 태스크만 Clear 하면 그 자리부터 이어서 실행됩니다.

새 DAG 파일은 dags 폴더에 넣으면 dag-processor 가 읽어 등록합니다. 목록에 보이지 않으면 airflow dags list-import-errors 로 문법 오류가 있는지 먼저 확인합니다.
""",
    "minio_guide.txt": """MinIO 보관 안내

MinIO 는 S3 와 호환되는 오브젝트 저장소입니다. 우리 과정에서는 데이터 레이크의 저장소로 쓰며, 원본 문서를 고치지 않고 그대로 보관합니다.

문서는 버킷이라는 큰 통에 담습니다. 원본 문서는 raw-docs 버킷에 올리고, 같은 이름으로 다시 올리면 이전 파일을 덮어씁니다. 그래서 같은 문서를 여러 번 올려도 파일은 하나만 남습니다.

버킷의 내용은 mc ls 명령으로 확인합니다. 웹 콘솔은 9001 포트에서 열리고 프로그램은 9000 포트의 API 로 접속합니다.

원본을 Lake 에 먼저 보관해 두면 임베딩 방법이나 조각 크기를 바꾸고 싶을 때 원본에서 다시 만들 수 있습니다.
""",
    "qdrant_guide.txt": """Qdrant 검색 안내

Qdrant 는 벡터 데이터베이스입니다. 문서를 작은 조각으로 자르고 조각마다 임베딩 벡터를 만들어 넣어 두면, 질문과 뜻이 가까운 조각을 찾아 줍니다.

조각을 넣는 단위는 포인트이고 포인트의 묶음을 컬렉션이라고 부릅니다. 컬렉션을 만들 때 정한 벡터 차원과 넣는 벡터의 차원이 다르면 오류가 납니다.

포인트의 id 를 문서 이름과 조각 순번으로 만들면 같은 문서를 다시 넣어도 같은 자리에 덮어씁니다. 무작위 id 를 쓰면 넣을 때마다 조각이 중복으로 쌓입니다.

검색 결과에는 0 에서 1 사이의 유사도 점수가 붙습니다. 점수가 낮은 결과는 질문과 관계없는 조각이므로 기준 점수보다 낮으면 답하지 않는 것이 안전합니다.
""",
    "lakehouse.txt": """레이크하우스 안내

레이크하우스는 데이터 레이크와 데이터 웨어하우스의 장점을 합친 구조입니다. 값싼 오브젝트 저장소에 파일을 두면서도 표처럼 다룰 수 있습니다.

우리 과정에서는 MinIO 위에 Iceberg 표 형식을 얹어 레이크하우스를 만들었습니다. Iceberg 는 어떤 파일이 표에 속하는지와 표의 변경 이력을 기록합니다.

같은 Iceberg 표를 Spark 와 Trino 두 엔진으로 읽을 수 있습니다. Spark 로 적재하고 Trino 로 빠르게 조회하는 식으로 역할을 나눕니다.

스냅샷이 남기 때문에 과거 시점의 데이터를 다시 조회할 수 있고, 잘못 적재했을 때 이전 스냅샷으로 되돌릴 수 있습니다.
""",
}

shutil.rmtree(kb.QDRANT, ignore_errors=True)          # 다시 실행해도 같은 결과가 되게 비우고 시작
c = kb.client()
for name, text in DOCS.items():
    print(f"{name:20s} {kb.index_doc(c, name, text)}조각")
print("지식베이스 전체", sum(kb.stat(c).values()), "조각")
c.close()

## 예제 1 · 검색 결과를 눈으로 보기

챗봇의 답은 검색 결과에서 나옵니다. 화면을 만들기 전에 **무엇이 몇 점으로 찾아지는지**부터 봅니다. (첫 실행은 임베딩 모델을 내려받느라 30초쯤 걸립니다.)

In [ ]:
def show(question, k=3):
    print("질문:", question)
    for score, source, text in kb.search(question, k):
        print(f"   {score:<6} {source:18s} {text[:44]}…")
    print()


show("스케줄러가 멈추면 어떻게 되나요?")
show("같은 문서를 다시 넣으면 조각이 중복되나요?")
show("김치찌개 끓이는 법 알려줘")

***확인 포인트***: 앞의 두 질문은 1위가 각각 `airflow_ops.txt`(0.476) · `minio_guide.txt`(0.603) 입니다. 「김치찌개」 질문에도 **결과 셋이 그대로 나옵니다**. 다만 1위 점수가 0.157 입니다.

벡터 검색은 「가까운 순서」로 줄을 세울 뿐, 「관계없음」이라고 말해 주지 않습니다. 관계없다고 판단하는 것은 우리가 정하는 **기준 점수**의 몫입니다.

**바꿔 보기**
- `show("…", k=5)` 로 다섯 개까지 봅니다. 4 · 5위의 점수는 1위와 얼마나 차이 나나요?

## 예제 2 · 기준 점수 정하기 — 문서 안 질문과 밖 질문

기준은 감으로 정하지 않고 **재서** 정합니다. 문서에 답이 있는 질문과 없는 질문을 여러 개 넣어 1위 점수를 표로 만듭니다.

In [ ]:
import pandas as pd

INSIDE = ["스케줄러가 멈추면 어떻게 되나요?",
          "실패한 태스크만 다시 돌리려면?",
          "원본 문서는 어디에 보관하나요?",
          "같은 문서를 다시 넣으면 조각이 중복되나요?",
          "레이크하우스가 뭔가요?",
          "Iceberg 표를 읽을 수 있는 엔진은?"]
OUTSIDE = ["김치찌개 끓이는 법 알려줘",
           "오늘 서울 날씨 어때?",
           "축구 경기는 몇 명이 뛰나요?"]
TRICKY = ["데이터 레이크와 데이터 마트의 차이는?"]      # 주제는 가까운데 문서에 답이 없는 질문

rows = []
for kind, questions in (("문서 안", INSIDE), ("문서 밖", OUTSIDE), ("까다로움", TRICKY)):
    for q in questions:
        score, source, _ = kb.search(q, 1)[0]
        rows.append({"종류": kind, "질문": q, "1위 점수": score, "1위 문서": source})
table = pd.DataFrame(rows)
print(table.to_string(index=False))

low_in = table[table["종류"] == "문서 안"]["1위 점수"].min()
high_out = table[table["종류"] == "문서 밖"]["1위 점수"].max()
print(f"\n문서 안 질문의 최저 {low_in} · 문서 밖 질문의 최고 {high_out}")
print("두 값 사이에 기준을 둘 수 있나요? →", "예" if low_in > high_out else "아니오 — 겹칩니다")

***확인 포인트***: 문서 안 질문은 최저 **0.441**, 문서 밖 질문은 최고 **0.276** 입니다. 두 값 사이가 비어 있어 `kb.py` 의 `THRESHOLD = 0.40` 이 둘을 가릅니다.

그런데 「까다로움」 줄을 봅니다. 「데이터 레이크와 데이터 마트의 차이는?」은 문서에 답이 없는데도 **0.657** 로, 문서 안 질문보다 높습니다. 「데이터 레이크」라는 낱말이 문서에 있어서입니다.
기준 점수가 거르는 것은 **주제가 다른 질문**입니다. 주제는 같은데 문서에 답이 없는 질문은 통과합니다. 이 질문을 막는 것은 다음 예제의 지시문(「근거에 없는 내용은 말하지 않습니다」)과, 답 아래에 근거를 보여 사람이 확인하게 하는 일입니다.

기준은 문서와 임베딩 모델이 바뀌면 달라집니다. 내 문서로 바꾸면 이 표부터 다시 만듭니다.

**바꿔 보기**
- `INSIDE` · `OUTSIDE` 에 질문을 두 개씩 더해 다시 잽니다. 두 값 사이가 좁아지나요?
- 문서 안 질문을 문서의 낱말을 쓰지 않고 돌려 말해 봅니다(예: 「일이 대기 줄에서 안 빠져요」). 점수가 얼마나 떨어지나요?

## 예제 3 · 답하는 함수 — 기준 미달이면 AI 를 부르지 않는다

답을 만드는 순서입니다.

1. 질문과 가까운 조각을 찾는다
2. 1위 점수가 기준보다 낮으면 **여기서 끝** — 「찾지 못했습니다」
3. 기준 이상이면 그 조각을 근거로 답한다 (OpenAI 키가 있으면 문장으로, 없으면 찾은 조각을 그대로)

2번에서 끝나면 AI 를 부르지 않으므로 **비용이 들지 않고, 지어낼 기회도 없습니다**.

코랩 보안 비밀에 `OPENAI_API_KEY` 가 있으면 아래 칸을 실행할 때 「노트북에 보안 비밀 액세스 권한이 없습니다」 창이 뜹니다. 키를 쓰려면 「액세스 권한 부여」, 키 없이 하려면 「취소」를 누릅니다. 어느 쪽이든 끝까지 돕니다.

In [ ]:
# OpenAI 키는 선택입니다 — 코랩 왼쪽 🔑 보안 비밀에 OPENAI_API_KEY 를 넣으면 문장으로 답합니다
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
except Exception:
    pass
HAS_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print("OpenAI 키:", "있음 — 문장으로 답합니다" if HAS_KEY else "없음 — 찾은 조각을 그대로 보여 줍니다")

CALLS = {"llm": 0}                          # AI 를 몇 번 불렀는지 셉니다


def respond(question, threshold=None):
    threshold = kb.THRESHOLD if threshold is None else threshold
    hits = kb.search(question, 3)
    top = hits[0][0] if hits else 0
    if top < threshold:                     # 근거가 없으면 여기서 끝냅니다
        return f"문서에서 찾지 못했습니다. (가장 가까운 조각의 점수 {top} < 기준 {threshold})"

    used = [h for h in hits if h[0] >= threshold][:2]
    context = "\n\n".join(f"[{source}] {text}" for _, source, text in used)
    refs = " · ".join(f"{source}({score})" for score, source, _ in used)
    if HAS_KEY:
        from openai import OpenAI
        CALLS["llm"] += 1
        r = OpenAI().chat.completions.create(
            model="gpt-4o-mini", temperature=0,
            messages=[{"role": "system", "content": "아래 근거만 써서 한국어 두세 문장으로 답합니다. 근거에 없는 내용은 말하지 않습니다."},
                      {"role": "user", "content": f"근거:\n{context}\n\n질문: {question}"}])
        body = r.choices[0].message.content
    else:
        body = used[0][2]
    return f"{body}\n\n근거: {refs}"


for q in ["스케줄러가 멈추면 어떻게 되나요?", "레이크하우스가 뭔가요?", "김치찌개 끓이는 법 알려줘", TRICKY[0]]:
    print("질문:", q)
    print(respond(q))
    print()
print("AI 를 부른 횟수:", CALLS["llm"])

***확인 포인트***: 앞의 두 질문에는 답과 함께 **근거 문서 이름과 점수**가 붙고, 「김치찌개」 질문에는 「문서에서 찾지 못했습니다」가 나옵니다.
키가 있다면 「AI 를 부른 횟수」가 4 가 아니라 **3** 입니다. 「김치찌개」 질문은 AI 까지 가지 않았습니다.

넷째(까다로운 질문)는 기준을 넘어 `lakehouse.txt` 가 근거로 붙습니다. 키가 없으면 레이크하우스 문단이 그대로 나오는데, 읽어 보면 「데이터 마트」 이야기는 없습니다. 키가 있으면 AI 가 지시문에 따라 근거에 없는 부분은 말하지 않는지 확인합니다. **근거를 함께 보여 주는 이유**가 여기 있습니다 — 읽는 사람이 답과 근거가 맞는지 볼 수 있습니다.

**바꿔 보기**
- `used = … [:2]` 를 `[:1]` 로 바꿔 근거를 하나만 쓰게 합니다. 「실패한 태스크만 다시 돌리려면?」의 답이 달라지나요?
- 시스템 지시문의 「근거에 없는 내용은 말하지 않습니다」를 지우면 무엇이 달라질지 예상해 봅니다(키가 있으면 직접 해 봅니다).

## 예제 4 · 일부러 기준을 없애기 — 엉뚱한 근거가 붙는다

기준을 0 으로 두면 어떤 질문이든 통과합니다. 문서 밖 질문 셋을 그대로 넣어 봅니다.

In [ ]:
for q in OUTSIDE:
    print("질문:", q)
    print(respond(q, threshold=0).replace("\n\n", "\n   "))
    print()

***확인 포인트***: 「김치찌개」 · 「날씨」 · 「축구」 질문에 Airflow 나 Qdrant 문단이 **근거라며** 붙어 나옵니다. 점수는 0.157 · 0.127 · 0.276 입니다.
키가 있다면 AI 가 「근거에 해당 내용이 없습니다」라고 답할 수도 있습니다. 그래도 비용은 이미 들었고, 근거 표시에는 관계없는 문서가 찍힙니다.

근거를 보여 주는 챗봇은 **근거가 틀리면 신뢰를 한 번에 잃습니다**. 기준 점수는 답의 품질 장치이면서 비용 장치입니다.

**바꿔 보기**
- 반대로 기준을 `0.7` 로 높여 `INSIDE` 질문을 넣어 봅니다. 답할 수 있는 질문인데 「찾지 못했습니다」가 몇 개 나오나요? 기준이 너무 높아도 쓸모가 없어집니다.

## 예제 5 · 화면 붙이기 — 대화형 챗봇

`respond` 함수에 화면을 붙입니다. `gr.ChatInterface` 는 질문 칸과 대화 창을 한 번에 만들어 줍니다(2장).
실행하면 아래에 채팅 화면이 뜨고 `https://….gradio.live` 주소가 나옵니다. 이 주소는 휴대폰에서도 열립니다(72시간 동안).

In [ ]:
import gradio as gr


def chat(message, history):
    return respond(message)


demo = gr.ChatInterface(
    chat,
    title="강의 자료 챗봇",
    description=f"문서 {len(DOCS)}개를 근거로 답합니다. 근거가 없으면 모른다고 답합니다(기준 {kb.THRESHOLD}).",
    examples=["스케줄러가 멈추면 어떻게 되나요?", "레이크하우스가 뭔가요?", "김치찌개 끓이는 법 알려줘"],
)
demo.launch(share="google.colab" in sys.modules)

***확인 포인트***: 채팅 화면이 뜨면 예시 질문 셋을 눌러 봅니다. 앞의 둘은 근거가 붙은 답, 셋째는 「문서에서 찾지 못했습니다」가 나옵니다. 예제 3 에서 글자로 본 것과 같은 답입니다 — 화면은 함수를 감싼 것뿐입니다.

화면을 끄려면 `demo.close()` 를 실행합니다.

**바꿔 보기**
- 문서를 내 것으로 바꿉니다. 준비 3 의 `DOCS` 에 내 업무 문서를 하나 더하고 그 칸부터 다시 실행한 뒤, **예제 2 의 표부터 다시 만들어** 기준이 여전히 맞는지 확인합니다.
- `chat` 함수가 `history`(이전 대화)를 받지만 쓰지 않습니다. 「그럼 그건 어디서 확인해요?」 같은 이어지는 질문에 약한 이유입니다. 직전 질문을 이번 질문 앞에 붙여 검색하게 고쳐 봅니다.

## 정리 — 다음 노트북으로 이어지는 것

오늘 만든 것은 「지식베이스가 이미 있을 때」의 챗봇입니다. 문서가 바뀌면 누가 지식베이스를 다시 만들까요? kb_01 의 파이프라인입니다.
kb_03 에서는 둘을 이어, **문서를 넣으면 다음 날 챗봇의 답이 바뀌는** 흐름을 끝까지 돌립니다.

## 마무리
이 노트북은 「코랩으로 만드는 업무 챗봇」 17장의 예제입니다. 「바꿔 보기」에서 고친 것과 달라진 결과를 한두 줄로 적어 두면, 내 문서로 바꿀 때 길잡이가 됩니다.